In [ ]:
###
# 1. 必要なライブラリのインポート
###

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

# GPUが使える場合はGPUを、使えない場合はCPUを使用する設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

In [ ]:
###
# 2. データの読み込みと前処理（Data Augmentationはまだ入れない）
###

# 画像サイズを統一し、PyTorchのテンソルに変換
# （128x128は、学習速度と精度のバランスが良いサイズです）
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# フォルダから画像を読み込み
data_dir = "/content/drive/MyDrive/cnn-hands-on/data/cats_vs_dogs"
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# クラス名の確認（['Cat', 'Dog'] になっているはずです）
print(f"クラス: {full_dataset.classes}")
print(f"総画像枚数: {len(full_dataset)}枚")

# 全データ（約2.3万枚）を学習用(80%)とテスト用(20%)に分割
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# データローダーの作成（バッチサイズ32で取り出す）
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"学習用: {len(train_dataset)}枚, テスト用: {len(test_dataset)}枚")

In [ ]:
###
# 3. AIの脳みそ（CNN）の設計図
###

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 第2回で学ぶ：畳み込み層
        # 入力3チャンネル(RGB) -> 出力16チャンネルへ特徴を抽出
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        # 入力16チャンネル -> 出力32チャンネルへさらに深く抽出
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # 第4回で学ぶ：全結合層
        # 128x128の画像が2回のプーリング(1/2)で 32x32 になる
        # 32(チャンネル) * 32(縦) * 32(横) = 32768
        self.fc1 = nn.Linear(in_features=32 * 32 * 32, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=2) # 最終出力は2クラス（犬/猫）

    def forward(self, x):
        # 第3回・第4回で学ぶ：データの流れ（順伝播）
        # 畳み込み -> ReLU(活性化) -> MaxPool(圧縮) のセット
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        
        # 1列に並べ直す（Flatten）
        x = torch.flatten(x, 1)
        
        # 全結合層
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        
        return x

# モデルの実体化とデバイス転送
model = SimpleCNN().to(device)
print(model)

In [ ]:
###
# 4. 学習の実行（AIを鍛える）
###

# 第5回で学ぶ：損失関数と最適化
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5 # 授業時間の都合上、まずは5エポック程度で回します
train_loss_list = []

print("🚀 学習をスタートします！")
for epoch in range(num_epochs):
    model.train() # 学習モード
    running_loss = 0.0
    
    # バッチごとにデータを取り出して学習
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # 第6回で学ぶ：学習の4ステップ
        optimizer.zero_grad()               # 1. 勾配のリセット
        outputs = model(images)             # 2. 予測（順伝播）
        loss = criterion(outputs, labels)   # 3. 誤差の計算
        loss.backward()                     # 4. 誤差逆伝播
        optimizer.step()                    # 5. パラメータ更新
        
        running_loss += loss.item()
        
    # 1エポック分の平均Lossを記録
    epoch_loss = running_loss / len(train_loader)
    train_loss_list.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("✅ 学習が完了しました！")

# Lossのグラフを表示
plt.plot(train_loss_list)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
###
# 5. 完成したモデルの保存
###

# modelsフォルダを作成して保存
os.makedirs('/content/drive/MyDrive/cnn-hands-on/models', exist_ok=True)
save_path = '/content/drive/MyDrive/cnn-hands-on/models/simple_cnn_cats_dogs.pth'

torch.save(model.state_dict(), save_path)
print(f"🎉 モデルを保存しました！: {save_path}")